# __MODEL_LABEL_MARKDOWN__

The visible cells own the model, data, transforms, validation, market edits,
and deployment decision. The imported helpers record generated identities,
versions, hashes, lineage, and rating-package rows.


In [ ]:
DATABASE_MODE = "local"  # "local" or "remote"
RUNTIME_MODULE = None  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = ""
ALLOW_REMOTE_WRITES = False

DATA_AS_OF = None  # Or use MODEL.data_as_of_column below.
SCORING = ("deviance", "nll", "gini")
RUN_EDITOR = False
EDIT_REASON = ""
DEPLOY = False
DEPLOYMENT_REASON = ""


In [ ]:
from datetime import date
from pathlib import Path
import sys

search_root = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in (search_root, *search_root.parents)
        if (root / "pricing_pipeline").is_dir()
        and (root / "pricing_models").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the pricing repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from superglm import Numeric, SuperGLM  # noqa: E402
from superglm.editor import EditorSession  # noqa: E402

from pricing_pipeline.models.config import ValidationSplitConfig  # noqa: E402
from pricing_pipeline.notebook import (  # noqa: E402
    PricingModelSpec,
    build_candidate,
    connect,
    deploy_package,
    open_candidate,
    publish_candidate,
    publish_edits,
    register_model,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"


## Define the model and validation decision

Features, scoring, validation, offsets, and independent fit/export weights remain visible Python.


In [ ]:
FEATURES = {
    "__FEATURE_NAME__": Numeric(),
}
superglm_model = SuperGLM(
    family="poisson",
    selection_penalty=0.0,
    discrete=True,
    n_bins=64,
    features=FEATURES,
)

MODEL = PricingModelSpec(
    name="__MODEL_NAME__",
    label="__MODEL_LABEL__",
    target="__TARGET_NAME__",
    model_type="__MODEL_TYPE__",
    deployment_slot="__DEPLOYMENT_SLOT__",
    features=tuple(FEATURES),
    dataset_name="__DATASET_NAME__",
    source_system="replace_with_source_name",
    pk_columns=("__PRIMARY_KEY__",),
    offset_column=None,
    offset_source_column=None,
    offset_label=None,
    sample_weight_column=None,
    export_weight_column=None,
    # For a visible term transform in the frame cell, replace the first three values with:
    # offset_column="term_offset",
    # offset_source_column="term",
    # offset_label="log(term / 12)",
    data_as_of_column="data_as_of",
    validation=ValidationSplitConfig.kfold(
        n_splits=5,
        random_state=42,
        shuffle=True,
    ),
    scoring=SCORING,
)


## Connect, verify the destination, and register the model

Local mode creates persistent SQLite files under `.local`. Remote mode uses the private runtime configured outside this repository and refuses writes until the expected database matches.


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display({
    "Destination": pricing.destination,
    "Artifact root": str(pricing.settings.workbench_artifact_root),
})
model = register_model(pricing, MODEL, source_root=MODEL_DIR)


## Load and transform the model frame

Replace the demo with the normal work query. Keep transforms as ordinary Python and retain every primary-key, target, offset/source, weight, split, and data-as-of column named in `MODEL`.


In [ ]:
rng = np.random.default_rng(42)
frame = pd.DataFrame({
    "__PRIMARY_KEY__": np.arange(1, 101),
    "__FEATURE_NAME__": rng.normal(size=100),
    "data_as_of": [date.today()] * 100,
})
frame["__TARGET_NAME__"] = rng.poisson(
    np.exp(-0.5 + 0.25 * frame["__FEATURE_NAME__"])
)

# If this model has an offset, make the transform visible here and name both columns above:
# frame["term_offset"] = np.log(frame["term"] / 12.0)
display({"Rows": len(frame), "Columns": len(frame.columns)})


## Fit the candidate


In [ ]:
candidate = build_candidate(
    pricing,
    model=model,
    frame=frame,
    superglm_model=superglm_model,
    data_as_of=DATA_AS_OF,
)


## Inspect held-out validation before any SQL publication


In [ ]:
display(candidate.validation_metrics)


## Publish the immutable baseline candidate

Publication records the audit trail and creates a package. It does not change the live deployment.


In [ ]:
published = publish_candidate(pricing, candidate)
display({
    "Model": published.model_name,
    "Model version": published.model_version,
    "Package": published.package_version,
    "State": published.package_status,
})


## Optional market edit: open the package and create a visible editor session

Remote mode only. Enable `RUN_EDITOR`, run this cell, and make the market changes in the displayed SuperGLM widget.


In [ ]:
reviewed = None
editor_session = None
editor_widget = None
if RUN_EDITOR:
    reviewed = open_candidate(
        pricing,
        model=model,
        package_version=published.package_version,
    )
    editor_session = EditorSession.from_model(
        reviewed.bundle.fitted_model,
        train_data=(
            reviewed.bundle.X,
            reviewed.bundle.y,
            reviewed.bundle.sample_weight,
            reviewed.bundle.offset,
        ),
        cv_report=reviewed.bundle.cv_report,
    )
    editor_widget = editor_session.widget()
    display(editor_widget)


## Materialize an in-memory preview of the edits

This ordinary SuperGLM action does not save, publish, deploy, or reopen anything.


In [ ]:
edited_model = None
if RUN_EDITOR:
    edited_model = editor_session.to_model()
edited_model


## Publish the editor session as an immutable child package


In [ ]:
edited = None
if RUN_EDITOR:
    if not EDIT_REASON.strip():
        raise ValueError("Describe the market or underwriting edit.")
    edited = publish_edits(
        pricing,
        candidate=reviewed,
        editor_session=editor_session,
        reason=EDIT_REASON,
    )
    reviewed = open_candidate(
        pricing,
        model=model,
        package_version=edited.package_version,
    )
    display({"Edited package": edited.package_version, "State": edited.package_status})


## Optional deployment of the reviewed package (remote mode only)


In [ ]:
if DEPLOY:
    if reviewed is None:
        reviewed = open_candidate(
            pricing,
            model=model,
            package_version=published.package_version,
        )
    if not DEPLOYMENT_REASON.strip():
        raise ValueError("Describe the approval for changing the live package.")
    deployment = deploy_package(
        pricing,
        package=reviewed,
        reason=DEPLOYMENT_REASON,
    )
    display(deployment)
